In [ ]:
"""
Данный файл берет на вход csv файлы полученные через запуск jupyter/dataset_intro.ipynb и запуск data/build_data.py
И создает 2 готовых датасета questions_with_chunks и questions_paraphrased_with_chunks для тестирования RAG
"""


%cd ..
import ast
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoModel, AutoModelForCausalLM, AutoTokenizer

/home/kalisto/Documents/9_sem/BigDataAndML/Lossy-RAG


In [2]:
questions = pd.read_csv("data/questions.csv")
questions_paraphrased = pd.read_csv("data/questions_paraphrased.csv")
contexts = pd.read_csv("data/contexts.csv")
answers = pd.read_csv("data/answers.csv")
chunks = pd.read_csv("data/chunks.csv")

In [3]:
questions = questions[questions["role"] == "valid"]
questions_paraphrased = questions_paraphrased[questions_paraphrased["role"] == "valid"]

questions.reset_index()
questions_paraphrased.reset_index()
answers.reset_index()

answers = answers.groupby("orig_qid", as_index=False).agg(list)


In [4]:
questions = questions[questions["orig_qid"].isin(questions_paraphrased["orig_qid"])]
questions = questions[~questions["is_paraphrase"]]

questions = pd.merge(questions, answers, on="orig_qid", how="left")
questions_paraphrased = pd.merge(questions_paraphrased, answers, on="orig_qid", how="left")

questions = pd.merge(questions, contexts, on="pid", how="left")
questions_paraphrased = pd.merge(questions_paraphrased, contexts, on="pid", how="left")

questions = questions.rename(columns={"text": "answer_text", "start": "answer_start_pos"})
questions_paraphrased = questions_paraphrased.rename(columns={"text": "answer_text", "start": "answer_start_pos"})

In [5]:
questions = questions.sort_values("orig_qid").reset_index(drop=True)
questions_paraphrased = questions_paraphrased.sort_values("orig_qid").reset_index(drop=True)

questions = questions.drop(columns=["squad_id"])
questions_paraphrased = questions_paraphrased.drop(columns=["squad_id"])

In [6]:
questions

,qid,orig_qid,role,is_paraphrase,pid,question,answer_text,answer_start_pos,title,context,n_chunks
0,88027,88027,valid,False,18914,Where was the practice place the Panthers used...,"[San Jose, the San Jose State practice facilit...","[22, 18, 22]",Super_Bowl_50,The Panthers used the San Jose State practice ...,1
1,88051,88051,valid,False,18916,What color were the footballs handed out to a ...,"[gold, Gold]","[8, 357]",Super_Bowl_50,Various gold-themed promotions and initiatives...,1
2,88112,88112,valid,False,18920,What is the name of the service that gets loca...,[Business Connect],[164],Super_Bowl_50,"For the first time, the Super Bowl 50 Host Com...",1
3,88115,88115,valid,False,18921,What is the name of the Host Committee's chari...,"[the 50 fund, 50 fund]","[200, 204]",Super_Bowl_50,The Super Bowl 50 Host Committee has vowed to ...,1
4,88144,88144,valid,False,18923,How many broadcast television partners does th...,[three],[84],Super_Bowl_50,"In the United States, the game was televised b...",3
...,...,...,...,...,...,...,...,...,...,...,...
495,98112,98112,valid,False,20945,What is needed to pack electrons densely toget...,"[energy, energy]","[579, 539]",Force,It is a common misconception to ascribe the st...,4
496,98116,98116,valid,False,20946,What is the force between nucleons?,"[nuclear, nuclear force., nuclear force]","[216, 216, 216]",Force,The strong force only acts directly upon eleme...,4
497,98131,98131,valid,False,20949,What is the final effect of adding more and mo...,"[conservation of mechanical energy, the tensio...","[997, 623, 627]",Force,Tension forces can be modeled using ideal stri...,6
498,98133,98133,valid,False,20950,What did Newton's mechanics affect?,"[idealized point particles, idealized point pa...","[100, 100]",Force,Newton's laws and Newtonian mechanics in gener...,3


In [7]:
questions_paraphrased

,qid,orig_qid,role,is_paraphrase,pid,question,answer_text,answer_start_pos,title,context,n_chunks
0,100417,88027,valid,True,18914,Where did the Panthers train ahead of the Supe...,"[San Jose, the San Jose State practice facilit...","[22, 18, 22]",Super_Bowl_50,The Panthers used the San Jose State practice ...,1
1,100463,88051,valid,True,18916,What hue were the balls given to many secondar...,"[gold, Gold]","[8, 357]",Super_Bowl_50,Various gold-themed promotions and initiatives...,1
2,100173,88112,valid,True,18920,Which program connects area firms with deals t...,[Business Connect],[164],Super_Bowl_50,"For the first time, the Super Bowl 50 Host Com...",1
3,100518,88115,valid,True,18921,What is the Host Committee's philanthropic pro...,"[the 50 fund, 50 fund]","[200, 204]",Super_Bowl_50,The Super Bowl 50 Host Committee has vowed to ...,1
4,100467,88144,valid,True,18923,With how many TV networks does the NFL have ai...,[three],[84],Super_Bowl_50,"In the United States, the game was televised b...",3
...,...,...,...,...,...,...,...,...,...,...,...
495,100330,98112,valid,True,20945,What is required to squeeze negatively charged...,"[energy, energy]","[579, 539]",Force,It is a common misconception to ascribe the st...,4
496,100287,98116,valid,True,20946,What interaction acts between protons and neut...,"[nuclear, nuclear force., nuclear force]","[216, 216, 216]",Force,The strong force only acts directly upon eleme...,4
497,100582,98131,valid,True,20949,What ultimately results from attaching ever mo...,"[conservation of mechanical energy, the tensio...","[997, 623, 627]",Force,Tension forces can be modeled using ideal stri...,6
498,100398,98133,valid,True,20950,To what objects did the physics of Newton apply?,"[idealized point particles, idealized point pa...","[100, 100]",Force,Newton's laws and Newtonian mechanics in gener...,3


In [8]:
from src.encoder import BaseEncoder
from src.quant.sq import SqQuant
from src.search.ivf import IvfSearch

texts = chunks["text"].drop_duplicates().tolist()

device = "cuda" if torch.cuda.is_available() else "cpu"
encoder_name = "BAAI/bge-large-en-v1.5"

encoder = BaseEncoder(
    model=AutoModel.from_pretrained(encoder_name).to(device),
    tokenizer=AutoTokenizer.from_pretrained(encoder_name),
    query_instruction="Represent this sentence for searching relevant passages: ",
)
searcher = IvfSearch(
    encoder=encoder,
    quantizer=SqQuant(num_bits=8),
    nlist=int(len(texts) ** 0.5),
    nprobe=4,
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [9]:
#1200 sec to fit
#database = Database(searcher)
#database.fit(texts)
#database.set_data(texts)


#searcher.encoder = None
#torch.save(database, f"models/database_{encoder_name.split('/')[-1]}.pt")
#searcher.encoder = encoder

In [10]:
database = torch.load(
    f"models/database_{encoder_name.split('/')[-1]}.pt",
    weights_only=False,
    map_location=device,
)
database.searcher.encoder = encoder

In [11]:
k = 10
num_threads = 20

run_dir = Path(f"data/{encoder_name.split('/')[-1]}_{k}")
run_dir.mkdir(parents=True, exist_ok=True)

with ThreadPoolExecutor(num_threads) as pool:
    questions["chunks"] = list(tqdm(
        pool.map(lambda q: database.search(q, k=k)["text"].tolist(), questions["question"]),
        total=len(questions),
    ))
questions.to_csv(f"{run_dir}/questions_with_chunks.csv", index=False)

with ThreadPoolExecutor(num_threads) as pool:
    questions_paraphrased["chunks"] = list(tqdm(
        pool.map(lambda q: database.search(q, k=k)["text"].tolist(), questions_paraphrased["question"]),
        total=len(questions_paraphrased),
    ))
questions_paraphrased.to_csv(f"{run_dir }/questions_paraphrased_with_chunks.csv", index=False)

  0%|          | 0/500 [00:00<?, ?it/s]

  0%|          | 0/500 [00:00<?, ?it/s]

In [12]:
run_dir = Path(f"data/{encoder_name.split('/')[-1]}_{k}")

questions = pd.read_csv(f"{run_dir}/questions_with_chunks.csv")
questions_paraphrased = pd.read_csv(f"{run_dir}/questions_paraphrased_with_chunks.csv")

for col in ["answer_text", "answer_start_pos"]:
    questions[col] = questions[col].apply(ast.literal_eval)
    questions_paraphrased[col] = questions_paraphrased[col].apply(ast.literal_eval)

In [13]:
questions

,qid,orig_qid,role,is_paraphrase,pid,question,answer_text,answer_start_pos,title,context,n_chunks,chunks
0,88027,88027,valid,False,18914,Where was the practice place the Panthers used...,"[San Jose, the San Jose State practice facilit...","[22, 18, 22]",Super_Bowl_50,The Panthers used the San Jose State practice ...,1,['The Panthers used the San Jose State practic...
1,88051,88051,valid,False,18916,What color were the footballs handed out to a ...,"[gold, Gold]","[8, 357]",Super_Bowl_50,Various gold-themed promotions and initiatives...,1,['Various gold-themed promotions and initiativ...
2,88112,88112,valid,False,18920,What is the name of the service that gets loca...,[Business Connect],[164],Super_Bowl_50,"For the first time, the Super Bowl 50 Host Com...",1,"['For the first time, the Super Bowl 50 Host C..."
3,88115,88115,valid,False,18921,What is the name of the Host Committee's chari...,"[the 50 fund, 50 fund]","[200, 204]",Super_Bowl_50,The Super Bowl 50 Host Committee has vowed to ...,1,"['The foundation section, based in Seattle, US..."
4,88144,88144,valid,False,18923,How many broadcast television partners does th...,[three],[84],Super_Bowl_50,"In the United States, the game was televised b...",3,['This deal included television rights for the...
...,...,...,...,...,...,...,...,...,...,...,...,...
495,98112,98112,valid,False,20945,What is needed to pack electrons densely toget...,"[energy, energy]","[579, 539]",Force,It is a common misconception to ascribe the st...,4,['When the electrons in a material are densely...
496,98116,98116,valid,False,20946,What is the force between nucleons?,"[nuclear, nuclear force., nuclear force]","[216, 216, 216]",Force,The strong force only acts directly upon eleme...,4,['The strong force only acts directly upon ele...
497,98131,98131,valid,False,20949,What is the final effect of adding more and mo...,"[conservation of mechanical energy, the tensio...","[997, 623, 627]",Force,Tension forces can be modeled using ideal stri...,6,['A computational problem can be viewed as an ...
498,98133,98133,valid,False,20950,What did Newton's mechanics affect?,"[idealized point particles, idealized point pa...","[100, 100]",Force,Newton's laws and Newtonian mechanics in gener...,3,"[""Newton's laws and Newtonian mechanics in gen..."


In [14]:
questions_paraphrased

,qid,orig_qid,role,is_paraphrase,pid,question,answer_text,answer_start_pos,title,context,n_chunks,chunks
0,100417,88027,valid,True,18914,Where did the Panthers train ahead of the Supe...,"[San Jose, the San Jose State practice facilit...","[22, 18, 22]",Super_Bowl_50,The Panthers used the San Jose State practice ...,1,['The Panthers used the San Jose State practic...
1,100463,88051,valid,True,18916,What hue were the balls given to many secondar...,"[gold, Gold]","[8, 357]",Super_Bowl_50,Various gold-themed promotions and initiatives...,1,['Various gold-themed promotions and initiativ...
2,100173,88112,valid,True,18920,Which program connects area firms with deals t...,[Business Connect],[164],Super_Bowl_50,"For the first time, the Super Bowl 50 Host Com...",1,"['For the first time, the Super Bowl 50 Host C..."
3,100518,88115,valid,True,18921,What is the Host Committee's philanthropic pro...,"[the 50 fund, 50 fund]","[200, 204]",Super_Bowl_50,The Super Bowl 50 Host Committee has vowed to ...,1,"[""The Congress of the South 40 (CS40) is a Res..."
4,100467,88144,valid,True,18923,With how many TV networks does the NFL have ai...,[three],[84],Super_Bowl_50,"In the United States, the game was televised b...",3,['This deal included television rights for the...
...,...,...,...,...,...,...,...,...,...,...,...,...
495,100330,98112,valid,True,20945,What is required to squeeze negatively charged...,"[energy, energy]","[579, 539]",Force,It is a common misconception to ascribe the st...,4,['This means that it takes energy to pack them...
496,100287,98116,valid,True,20946,What interaction acts between protons and neut...,"[nuclear, nuclear force., nuclear force]","[216, 216, 216]",Force,The strong force only acts directly upon eleme...,4,['The strong and weak forces are nuclear force...
497,100582,98131,valid,True,20949,What ultimately results from attaching ever mo...,"[conservation of mechanical energy, the tensio...","[997, 623, 627]",Force,Tension forces can be modeled using ideal stri...,6,['Sheave and rope wear may be significantly re...
498,100398,98133,valid,True,20950,To what objects did the physics of Newton apply?,"[idealized point particles, idealized point pa...","[100, 100]",Force,Newton's laws and Newtonian mechanics in gener...,3,"[""Newton's laws and Newtonian mechanics in gen..."


In [15]:
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import torch
from src.encoder import BaseEncoder
from src.quant.sq import SqQuant
from src.search.ivf import IvfSearch
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer

reranker_name = "Qwen/Qwen3-Reranker-0.6B"

rr_tokenizer = AutoTokenizer.from_pretrained(reranker_name, padding_side="left")
reranker = AutoModelForCausalLM.from_pretrained(reranker_name, dtype=torch.bfloat16).to(device).eval()
yes_id = rr_tokenizer.convert_tokens_to_ids("yes")
no_id = rr_tokenizer.convert_tokens_to_ids("no")

RR_PREFIX = (
    "<|im_start|>system\nJudge whether the Document meets the requirements based on the Query "
    'and the Instruct provided. Note that the answer can only be "yes" or "no".<|im_end|>\n'
    "<|im_start|>user\n"
)
RR_SUFFIX = "<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"
RR_INSTRUCT = "Given a web search query, retrieve relevant passages that answer the query"

k_retrieve = 30
k_rerank = 3
num_threads = 4


def search_rerank(q: str) -> list[str]:
    candidates = database.search(q, k=k_retrieve)["text"].tolist()
    pairs = [
        f"{RR_PREFIX}<Instruct>: {RR_INSTRUCT}\n<Query>: {q}\n<Document>: {c}{RR_SUFFIX}"
        for c in candidates
    ]
    inputs = rr_tokenizer(pairs, padding=True, return_tensors="pt").to(device)
    with torch.inference_mode():
        logits = reranker(**inputs, logits_to_keep=1).logits[:, -1]
    scores = logits[:, yes_id] - logits[:, no_id]
    return [candidates[i] for i in scores.topk(min(k_rerank, len(candidates))).indices.tolist()]


def retrieve(df: pd.DataFrame, path: str) -> None:
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with ThreadPoolExecutor(num_threads) as pool:
        df["chunks"] = list(tqdm(pool.map(search_rerank, df["question"]), total=len(df)))
    df.to_csv(path, index=False)


run_dir = Path(f"data/{encoder_name.split('/')[-1]}_{reranker_name.split('/')[-1]}_{k_retrieve}_{k_rerank}")
run_dir.mkdir(parents=True, exist_ok=True)

retrieve(questions, f"{run_dir}/questions_with_chunks.csv")
retrieve(questions_paraphrased, f"{run_dir}/questions_paraphrased_with_chunks.csv")

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

  0%|          | 0/500 [00:00<?, ?it/s]

  0%|          | 0/500 [00:00<?, ?it/s]

In [18]:
run_dir = Path(f"data/{encoder_name.split('/')[-1]}_{reranker_name.split('/')[-1]}_{k_retrieve}_{k_rerank}")

questions = pd.read_csv(f"{run_dir}/questions_with_chunks.csv")
questions_paraphrased = pd.read_csv(f"{run_dir}/questions_paraphrased_with_chunks.csv")

for col in ["answer_text", "answer_start_pos"]:
    questions[col] = questions[col].apply(ast.literal_eval)
    questions_paraphrased[col] = questions_paraphrased[col].apply(ast.literal_eval)

In [19]:
questions

,qid,orig_qid,role,is_paraphrase,pid,question,answer_text,answer_start_pos,title,context,n_chunks,chunks
0,88027,88027,valid,False,18914,Where was the practice place the Panthers used...,"[San Jose, the San Jose State practice facilit...","[22, 18, 22]",Super_Bowl_50,The Panthers used the San Jose State practice ...,1,['The Panthers used the San Jose State practic...
1,88051,88051,valid,False,18916,What color were the footballs handed out to a ...,"[gold, Gold]","[8, 357]",Super_Bowl_50,Various gold-themed promotions and initiatives...,1,['Various gold-themed promotions and initiativ...
2,88112,88112,valid,False,18920,What is the name of the service that gets loca...,[Business Connect],[164],Super_Bowl_50,"For the first time, the Super Bowl 50 Host Com...",1,"['For the first time, the Super Bowl 50 Host C..."
3,88115,88115,valid,False,18921,What is the name of the Host Committee's chari...,"[the 50 fund, 50 fund]","[200, 204]",Super_Bowl_50,The Super Bowl 50 Host Committee has vowed to ...,1,"[""Some NPOs which are particularly well known,..."
4,88144,88144,valid,False,18923,How many broadcast television partners does th...,[three],[84],Super_Bowl_50,"In the United States, the game was televised b...",3,"[""In the United States, the game was televised..."
...,...,...,...,...,...,...,...,...,...,...,...,...
495,98112,98112,valid,False,20945,What is needed to pack electrons densely toget...,"[energy, energy]","[579, 539]",Force,It is a common misconception to ascribe the st...,4,['When the electrons in a material are densely...
496,98116,98116,valid,False,20946,What is the force between nucleons?,"[nuclear, nuclear force., nuclear force]","[216, 216, 216]",Force,The strong force only acts directly upon eleme...,4,"['However, a residual of the force is observed..."
497,98131,98131,valid,False,20949,What is the final effect of adding more and mo...,"[conservation of mechanical energy, the tensio...","[997, 623, 627]",Force,Tension forces can be modeled using ideal stri...,6,['Adding 100 to the program counter would caus...
498,98133,98133,valid,False,20950,What did Newton's mechanics affect?,"[idealized point particles, idealized point pa...","[100, 100]",Force,Newton's laws and Newtonian mechanics in gener...,3,"[""Newton's laws and Newtonian mechanics in gen..."


In [ ]:
questions_paraphrased

,qid,orig_qid,role,is_paraphrase,pid,question,answer_text,answer_start_pos,title,context,n_chunks,chunks
0,100417,88027,valid,True,18914,Where did the Panthers train ahead of the Supe...,"[San Jose, the San Jose State practice facilit...","[22, 18, 22]",Super_Bowl_50,The Panthers used the San Jose State practice ...,1,['The Panthers used the San Jose State practic...
1,100463,88051,valid,True,18916,What hue were the balls given to many secondar...,"[gold, Gold]","[8, 357]",Super_Bowl_50,Various gold-themed promotions and initiatives...,1,['Various gold-themed promotions and initiativ...
2,100173,88112,valid,True,18920,Which program connects area firms with deals t...,[Business Connect],[164],Super_Bowl_50,"For the first time, the Super Bowl 50 Host Com...",1,"['For the first time, the Super Bowl 50 Host C..."
3,100518,88115,valid,True,18921,What is the Host Committee's philanthropic pro...,"[the 50 fund, 50 fund]","[200, 204]",Super_Bowl_50,The Super Bowl 50 Host Committee has vowed to ...,1,"[""Northwestern students also participate in th..."
4,100467,88144,valid,True,18923,With how many TV networks does the NFL have ai...,[three],[84],Super_Bowl_50,"In the United States, the game was televised b...",3,"[""In the United States, the game was televised..."
...,...,...,...,...,...,...,...,...,...,...,...,...
495,100330,98112,valid,True,20945,What is required to squeeze negatively charged...,"[energy, energy]","[579, 539]",Force,It is a common misconception to ascribe the st...,4,['When the electrons in a material are densely...
496,100287,98116,valid,True,20946,What interaction acts between protons and neut...,"[nuclear, nuclear force., nuclear force]","[216, 216, 216]",Force,The strong force only acts directly upon eleme...,4,['The strong and weak forces are nuclear force...
497,100582,98131,valid,True,20949,What ultimately results from attaching ever mo...,"[conservation of mechanical energy, the tensio...","[997, 623, 627]",Force,Tension forces can be modeled using ideal stri...,6,['Tension forces can be modeled using ideal st...
498,100398,98133,valid,True,20950,To what objects did the physics of Newton apply?,"[idealized point particles, idealized point pa...","[100, 100]",Force,Newton's laws and Newtonian mechanics in gener...,3,"[""Newton's laws and Newtonian mechanics in gen..."


: 